# cineQMAE: Data Preparation and Pretraining

Raw private/public CMR -> cine caches -> offline motion teachers -> MAE pretraining.
Edit only the paths, source and selected methods below. Numerical training/model/loss
defaults match the supplied notebook; no downstream training or signature audits.

`public` means **all patients from ACDC + MnM1 + MnM2**, including original test
images and NOR/unlabeled cases. This is transductive pretraining and must be
described as such in the paper. Private pretraining retains the historical
train-only split. Public monitor/snapshot cases are training diagnostics.


In [ ]:
from pathlib import Path
import sys

PACKAGE = Path.cwd().resolve()
sys.path.insert(0, str(PACKAGE))
from cmr_study import default_settings, prepare_data

SOURCE = "private"  # "private" or "public" (all three public datasets, all patients)
METHODS = ["videomae", "heartmae", "trackmae", "cineqmae"]
ROOT = Path("/public/home/chenweilin/CMRDataSet")

settings = default_settings()
settings["pretrain_source"] = SOURCE
settings["methods"] = METHODS
settings["private_root"] = str(ROOT)
settings["private_train_csv"] = "train_clean.csv"
settings["private_test_csv"] = "test_clean.csv"
settings["private_cache"] = "cmr_csv_cache_native_xy_v2"
settings["private_previous_manifest"] = None  # optional: exact historical split manifest
settings["public_raw_root"] = "/data/SAX"
settings["public_raw_roots"] = {"acdc": None, "mnm1": None, "mnm2": None}  # explicit extracted raw paths if needed
settings["public_cache_root"] = str(ROOT / "public_cmr_paper_cache")
settings["output_root"] = str(ROOT / "training_outputs")
settings["run_name"] = "cineq_paper_" + SOURCE
settings["motion"]["cache_root"] = str(ROOT / "cmr_study_motion_cache_native_v2")
settings["motion"]["cotracker_repo"] = str(PACKAGE / "vendor/cotracker")  # or your existing offline checkout
settings["motion"]["cotracker_checkpoint"] = str(PACKAGE / "weights/scaled_offline.pth")
settings["resume"] = False  # True to continue this run's existing last.pt

print("Source:", SOURCE, "| methods:", METHODS)
print("Pretraining:", settings["pretrain"])
print("Shared decoder:", settings["shared_decoder"])
print("AMP:", settings["amp"], "| TF32:", settings["allow_tf32"])


## 1. Raw Data to Caches

The selected source is processed without requiring the other source. To build
both caches now, set `PREPARE_SOURCES = ["private", "public"]`. Original files are
not modified. Cached patient manifests are reused on subsequent runs.


In [ ]:
PREPARE_SOURCES = [SOURCE]
prepared = prepare_data(settings, sources=PREPARE_SOURCES)
for name, patients in prepared.items():
    print(name, "patients:", len(patients))
    display(patients.groupby(["dataset", "split"]).size().rename("patients").reset_index())


## 2. Offline Teachers

Only teachers needed by the selected methods are prepared. TrackMAE needs your
offline CoTracker code/weights. Fixed snapshot references retain the previous
settings and include registration. No online teacher estimation occurs in training.


In [ ]:
from cmr_study import PretrainingExperiment

experiment = PretrainingExperiment(settings)
display(experiment.counts())
experiment.prepare_motion()

## 3. Pretraining

400 epochs, batch 32, accumulation 1, FP32; all other defaults are retained.
Each method/seed writes `pretrain/<method>/seed<seed>/last.pt`, `encoder_last.pt`
and `history.csv`, plus the original 50-epoch snapshots. Existing checkpoints
are not overwritten unless you explicitly resume their training.


In [ ]:
results = experiment.pretrain()
display(results)
print("Output:", experiment.root)